# Experimental Models Tutorial - DeepTab v2.0

This notebook demonstrates how to use experimental models from `deeptab.models.experimental`.

**What are experimental models?**
- Cutting-edge architectures from recent research
- Same API as stable models
- Not covered by semantic versioning (may change in minor releases)
- Fully functional and production-ready (but pin versions!)

**Topics covered:**
- Importing experimental models
- Classification, regression, and LSS examples
- Version pinning best practices
- Switching to stable imports after promotion

**Requirements:**
```bash
pip install deeptab==2.0.0  # Pin exact version!
pip install scikit-learn pandas numpy
```

## 1. Import Paths: Stable vs Experimental

In [ ]:
# Stable models - from deeptab.models
from deeptab.models import MambularClassifier, MambularRegressor, MambularLSS

# Experimental models - from deeptab.models.experimental
from deeptab.models.experimental import (
    TromptClassifier,
    ModernNCARegressor,
    TangosLSS,
)

print("✓ Stable imports:      deeptab.models")
print("✓ Experimental imports: deeptab.models.experimental")
print("\n⚠ Always pin DeepTab version when using experimental models!")

## 2. Classification with Experimental Model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from deeptab.models.experimental import TromptClassifier

# Set random seed
np.random.seed(42)

# Generate data
n_samples, n_features, n_classes = 1000, 6, 3
X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, n_classes, size=n_samples)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset: {len(X_train)} train, {len(X_test)} test")
print(f"Classes: {n_classes}")

In [ ]:
# Train experimental classifier
model = TromptClassifier()
model.fit(X_train, y_train, max_epochs=50)

In [ ]:
# Evaluate
metrics = model.evaluate(X_test, y_test)
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Loss: {metrics['loss']:.3f}")

# Predict
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

print(f"\nPredictions shape: {predictions.shape}")
print(f"Probabilities shape: {probabilities.shape}")
print(f"Sample predictions: {predictions[:5]}")

## 3. Regression with Experimental Model

In [ ]:
from deeptab.models.experimental import ModernNCARegressor

# Generate regression data
np.random.seed(42)
n_samples, n_features = 1000, 5
X = np.random.randn(n_samples, n_features)
y = X @ np.random.randn(n_features) + np.random.randn(n_samples) * 0.1

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

print(f"Regression dataset: {len(X_train)} train, {len(X_test)} test")

In [ ]:
# Train experimental regressor
model = ModernNCARegressor()
model.fit(X_train, y_train, max_epochs=50)

In [ ]:
# Evaluate
metrics = model.evaluate(X_test, y_test)
print(f"RMSE: {metrics['rmse']:.3f}")
print(f"MAE: {metrics['mae']:.3f}")
print(f"R²: {model.score(X_test, y_test):.3f}")

# Predict
predictions = model.predict(X_test)
print(f"\nPredictions: {predictions[:5]}")

## 4. LSS with Experimental Model

In [ ]:
from deeptab.models.experimental import TangosLSS

# Generate data for LSS
np.random.seed(42)
X = np.random.randn(1000, 5)
y = np.dot(X, np.random.randn(5)) + np.random.randn(1000)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(5)])
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42
)

print(f"LSS dataset: {len(X_train)} train, {len(X_test)} test")

In [ ]:
# Train experimental LSS model
model = TangosLSS()
model.fit(X_train, y_train, family="normal", max_epochs=50)

In [ ]:
# Get distribution parameters
params = model.predict(X_test)
print(f"Parameters shape: {params.shape}")
print("Column 0: mean, Column 1: log(std)")

# Extract mean and std
mean = params[:, 0]
log_std = params[:, 1]
std = np.exp(log_std)

print(f"\nMean of predicted means: {mean.mean():.3f}")
print(f"Mean of predicted stds: {std.mean():.3f}")

# Generate 90% prediction interval
from scipy import stats
lower = stats.norm.ppf(0.05, loc=mean, scale=std)
upper = stats.norm.ppf(0.95, loc=mean, scale=std)

coverage = np.mean((y_test >= lower) & (y_test <= upper))
print(f"\n90% interval coverage: {coverage:.3f}")

## 5. Customization with Configs

Experimental models support the same config system as stable models.

In [ ]:
from deeptab.configs import MambularConfig, PreprocessingConfig, TrainerConfig
from deeptab.models.experimental import TromptClassifier

# Configure model
model_cfg = MambularConfig(
    d_model=256,
    n_layers=8,
    dropout=0.3,
)

prep_cfg = PreprocessingConfig(
    numerical_preprocessing="quantile",
    use_ple=True,
)

trainer_cfg = TrainerConfig(
    lr=1e-3,
    batch_size=256,
    patience=15,
)

# Create model with configs
model_custom = TromptClassifier(
    model_config=model_cfg,
    preprocessing_config=prep_cfg,
    trainer_config=trainer_cfg,
)

# Train
model_custom.fit(X_train, y_train, max_epochs=100)

# Evaluate
metrics = model_custom.evaluate(X_test, y_test)
print(f"Custom model accuracy: {metrics['accuracy']:.3f}")

## 6. Integration with scikit-learn

Experimental models work with GridSearchCV and other sklearn tools.

In [ ]:
from sklearn.model_selection import GridSearchCV
from deeptab.models.experimental import TromptClassifier

param_grid = {
    "model_config__d_model": [128, 256],
    "model_config__n_layers": [4, 6],
    "trainer_config__lr": [5e-4, 1e-3],
}

model = TromptClassifier()

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=2,
)

print("Running grid search...")
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.3f}")

## 7. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score
from deeptab.models.experimental import ModernNCARegressor

model_cv = ModernNCARegressor()

scores = cross_val_score(
    model_cv, X_train, y_train,
    cv=5,
    scoring="neg_mean_squared_error",
)

rmse_scores = np.sqrt(-scores)
print(f"CV RMSE: {rmse_scores.mean():.3f} (+/- {rmse_scores.std():.3f})")

## 8. Available Experimental Models

List of experimental models in v2.0.

In [ ]:
from deeptab.models.experimental import (
    # Classification
    TromptClassifier,
    ModernNCAClassifier,
    TangosClassifier,
    
    # Regression
    TromptRegressor,
    ModernNCARegressor,
    TangosRegressor,
    
    # LSS (Distributional)
    TromptLSS,
    ModernNCALSS,
    TangosLSS,
)

experimental_models = {
    "Classification": ["TromptClassifier", "ModernNCAClassifier", "TangosClassifier"],
    "Regression": ["TromptRegressor", "ModernNCARegressor", "TangosRegressor"],
    "LSS": ["TromptLSS", "ModernNCALSS", "TangosLSS"],
}

print("Available Experimental Models:")
print("="*50)
for task, models in experimental_models.items():
    print(f"\n{task}:")
    for m in models:
        print(f"  - {m}")

## 9. Comparing Experimental and Stable Models

Same API - different import paths.

In [ ]:
from deeptab.models import MambularClassifier  # Stable
from deeptab.models.experimental import TromptClassifier  # Experimental

# Generate small dataset for quick comparison
X_small = np.random.randn(500, 5)
y_small = np.random.randint(0, 3, size=500)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_small, y_small, test_size=0.2, stratify=y_small, random_state=42
)

# Compare architectures
for ModelClass in [MambularClassifier, TromptClassifier]:
    print(f"\nTraining {ModelClass.__name__}...")
    model = ModelClass()
    model.fit(X_train_s, y_train_s, max_epochs=30)
    accuracy = model.score(X_test_s, y_test_s)
    print(f"  Accuracy: {accuracy:.3f}")

## 10. Version Pinning Best Practices

In [ ]:
import deeptab

print("Version Pinning Best Practices:")
print("="*50)
print("\n1. Check current version:")
print(f"   DeepTab version: {deeptab.__version__}")

print("\n2. Pin in requirements.txt:")
print("   deeptab==2.0.0")

print("\n3. Or in pyproject.toml:")
print('   dependencies = ["deeptab==2.0.0"]')

print("\n4. Why pin versions?")
print("   - Experimental APIs may change in minor releases")
print("   - Stable models follow semantic versioning")
print("   - Pinning prevents unexpected breakage")

print("\n5. Monitor release notes:")
print("   - Check for API changes before upgrading")
print("   - Update imports when models are promoted to stable")

## 11. Switching to Stable After Promotion

When an experimental model is promoted to stable, only the import changes.

In [ ]:
# Example: Before promotion
# from deeptab.models.experimental import TromptClassifier
# 
# model = TromptClassifier()
# model.fit(X_train, y_train, max_epochs=50)

# After promotion (announced in release notes):
# from deeptab.models import TromptClassifier  # Only change
# 
# model = TromptClassifier()
# model.fit(X_train, y_train, max_epochs=50)  # Everything else identical

print("When a model is promoted to stable:")
print("="*50)
print("✓ Only the import path changes")
print("✓ All code (fit, predict, configs, etc.) stays the same")
print("✓ Check release notes for promotion announcements")
print("✓ Update imports and remove version pin")

## 12. Model Promotion Criteria

What makes an experimental model graduate to stable?

In [ ]:
promotion_criteria = [
    "Performance - Competitive with existing stable models",
    "Stability - No known bugs or crashes",
    "Testing - Comprehensive unit and integration tests",
    "Documentation - Full API documentation and examples",
    "Community feedback - Positive user experience",
    "Production use - Successfully used in real-world projects",
]

print("Model Promotion Criteria:")
print("="*50)
for i, criterion in enumerate(promotion_criteria, 1):
    print(f"{i}. {criterion}")

print("\n✓ See developer_guide/model_promotion_policy for details")

## 13. Checking Model Tier at Runtime

In [ ]:
from deeptab.models import MambularClassifier
from deeptab.models.experimental import TromptClassifier

# Check if model is experimental
is_experimental_trompt = hasattr(TromptClassifier, "_experimental")
is_experimental_mambular = hasattr(MambularClassifier, "_experimental")

print("Runtime tier detection:")
print("="*50)
print(f"TromptClassifier is experimental: {is_experimental_trompt}")
print(f"MambularClassifier is experimental: {is_experimental_mambular}")

## 14. Save and Load Experimental Models

In [ ]:
from deeptab.models.experimental import TromptClassifier

# Train and save
model = TromptClassifier()
model.fit(X_train, y_train, max_epochs=30)
model.save("experimental_model.pkl")
print("✓ Experimental model saved")

# Load later (same import path required)
loaded_model = TromptClassifier.load("experimental_model.pkl")
predictions = loaded_model.predict(X_test)
print(f"✓ Model loaded, predictions: {predictions[:5]}")

## 15. When to Use Experimental Models?

In [ ]:
use_cases = {
    "✓ Use experimental models when": [
        "You want to try cutting-edge architectures",
        "You're willing to pin versions",
        "You can tolerate potential API changes",
        "You want to provide early feedback",
        "You're exploring different approaches",
    ],
    "⚠ Use stable models when": [
        "You need API stability guarantees",
        "You're in production without version pinning",
        "You want long-term support",
        "You need battle-tested reliability",
    ]
}

for category, points in use_cases.items():
    print(f"\n{category}")
    print("="*50)
    for point in points:
        print(f"  • {point}")

## Summary

In this tutorial, you learned how to:
- ✅ Import experimental models from `deeptab.models.experimental`
- ✅ Use experimental models for classification, regression, and LSS
- ✅ Customize with configs (same as stable models)
- ✅ Integrate with scikit-learn tools
- ✅ Pin versions to avoid breaking changes
- ✅ Switch to stable imports after promotion
- ✅ Understand model promotion criteria

**Key takeaways:**
- Experimental models have the same API as stable models
- Always pin DeepTab version (`deeptab==x.y.z`)
- Monitor release notes for promotions
- Only import path changes when promoted to stable

**Next steps:**
- Try [Classification Tutorial](classification.ipynb) with stable models
- Check [Regression Tutorial](regression.ipynb) for standard regressors
- Explore [Distributional Regression](distributional.ipynb) for uncertainty

**Documentation:** https://deeptab.readthedocs.io/